# Sentiment Analysis by Subreddit Clusters

This notebook analyzes sentiment at the **cluster level** using subreddit cluster assignments from `subreddit_cluster_summary_k20.csv`.

Each cluster groups multiple subreddits together. All documents from subreddits within a cluster are aggregated for that cluster's sentiment analysis.

**Workflow:**
1. Load `reddit_topic_assignments.csv` (documents with subreddit info)
2. Load `subreddit_cluster_summary_k20.csv` (cluster → subreddit mapping)
3. Map each document to its cluster via subreddit name
4. Run Twitter-RoBERTa sentiment model on all documents
5. Aggregate and visualize sentiment by cluster

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load Data and Create Cluster Mapping

In [ ]:
# Load the main document data
df = pd.read_csv('reddit_topic_assignments.csv')
print(f"Loaded {len(df)} documents")
print(f"Columns: {list(df.columns)}")
print(f"Unique subreddits in documents: {df['subreddit'].nunique()}")
print()

# Load the cluster summary
clusters_df = pd.read_csv('subreddit_cluster_summary_k20.csv')
print(f"Loaded {len(clusters_df)} clusters")
print(f"Columns: {list(clusters_df.columns)}")
print()
clusters_df.head()

In [ ]:
# Parse the subreddits column to create a subreddit -> cluster mapping
# The subreddits column contains semicolon-separated subreddit names
subreddit_to_cluster = {}
subreddit_to_cluster_name = {}

for _, row in clusters_df.iterrows():
    cluster_id = row['cluster']
    cluster_name = row['Name by LLM']
    subreddits_str = row['subreddits']
    
    # Split by semicolon and strip whitespace
    subreddits = [s.strip() for s in str(subreddits_str).split(';') if s.strip()]
    
    for sub in subreddits:
        subreddit_to_cluster[sub] = cluster_id
        subreddit_to_cluster_name[sub] = cluster_name

print(f"Total subreddit -> cluster mappings: {len(subreddit_to_cluster)}")
print(f"Number of clusters: {clusters_df['cluster'].nunique()}")
print()

# Map clusters onto the document dataframe
df['cluster_id'] = df['subreddit'].map(subreddit_to_cluster)
df['cluster_name'] = df['subreddit'].map(subreddit_to_cluster_name)

# Check coverage
matched = df['cluster_id'].notna().sum()
unmatched = df['cluster_id'].isna().sum()
print(f"Documents matched to a cluster: {matched} ({matched/len(df)*100:.1f}%)")
print(f"Documents with no cluster match: {unmatched} ({unmatched/len(df)*100:.1f}%)")

if unmatched > 0:
    unmatched_subs = df[df['cluster_id'].isna()]['subreddit'].unique()
    print(f"\nUnmatched subreddits ({len(unmatched_subs)}): {list(unmatched_subs)[:20]}")
    if len(unmatched_subs) > 20:
        print(f"   ... and {len(unmatched_subs) - 20} more")

In [ ]:
# Parse month/time column for time-based analysis
# Try multiple date formats to handle different possible formats
df['month_dt'] = pd.to_datetime(df['month'], errors='coerce')

# If all NaT, try other common formats
if df['month_dt'].isna().all():
    for fmt in ['%Y-%m', '%Y/%m', '%m/%Y', '%Y%m', '%b %Y', '%B %Y']:
        try:
            parsed = pd.to_datetime(df['month'], format=fmt, errors='coerce')
            if parsed.notna().any():
                df['month_dt'] = parsed
                print(f"Parsed month column with format: {fmt}")
                break
        except:
            continue

if df['month_dt'].notna().any():
    df['quarter'] = df['month_dt'].dt.to_period('Q').astype(str)
    print(f"Date range: {df['month_dt'].min()} to {df['month_dt'].max()}")
    print(f"Quarters found: {sorted(df['quarter'].dropna().unique())}")
else:
    print("WARNING: Could not parse 'month' column into dates.")
    print(f"Sample month values: {df['month'].dropna().unique()[:10]}")
    # Use the raw month column as-is for grouping
    df['quarter'] = df['month'].astype(str)
    print(f"Using raw 'month' values for time grouping: {sorted(df['quarter'].dropna().unique())[:10]}")

In [ ]:
# Summary of cluster assignments
print("Documents per cluster:\n")
cluster_counts = df.groupby(['cluster_id', 'cluster_name']).size().reset_index(name='n_docs')
cluster_counts = cluster_counts.sort_values('n_docs', ascending=False)
for _, row in cluster_counts.iterrows():
    print(f"  Cluster {int(row['cluster_id']):2d} | {row['cluster_name']:<50s} | {row['n_docs']:5d} docs")
print(f"\n  Total assigned: {cluster_counts['n_docs'].sum()} docs")

## 2. Sentiment Analysis with Twitter-RoBERTa

In [ ]:
class RedditSentimentAnalyzer:
    """Sentiment analyzer using Twitter-RoBERTa model"""

    def __init__(self, model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"):
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        self.labels = ['negative', 'neutral', 'positive']
        print(f"Model loaded successfully on {self.device}")

    def preprocess_text(self, text: str) -> str:
        if pd.isna(text):
            return ""
        text = str(text)
        new_text = []
        for t in text.split():
            t = '@user' if t.startswith('@') and len(t) > 1 else t
            t = 'http' if t.startswith('http') else t
            new_text.append(t)
        return " ".join(new_text)

    def analyze_sentiment(self, text: str) -> Dict[str, float]:
        text = self.preprocess_text(text)
        if not text or len(text.strip()) == 0:
            return {label: 0.0 for label in self.labels}
        encoded_input = self.tokenizer(
            text, return_tensors='pt', truncation=True, max_length=512, padding=True
        )
        encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}
        with torch.no_grad():
            output = self.model(**encoded_input)
        scores = output.logits[0].cpu().numpy()
        scores = softmax(scores)
        return {label: float(score) for label, score in zip(self.labels, scores)}

    def analyze_dataframe(self, df: pd.DataFrame, text_column: str = 'Document',
                         batch_size: int = 32) -> pd.DataFrame:
        print(f"Analyzing {len(df)} documents...")
        results = []
        for idx in tqdm(range(0, len(df), batch_size)):
            batch = df[text_column].iloc[idx:idx+batch_size]
            for text in batch:
                sentiment = self.analyze_sentiment(text)
                results.append(sentiment)
        sentiment_df = pd.DataFrame(results)
        df_with_sentiment = df.copy()
        df_with_sentiment['sentiment_negative'] = sentiment_df['negative'].values
        df_with_sentiment['sentiment_neutral'] = sentiment_df['neutral'].values
        df_with_sentiment['sentiment_positive'] = sentiment_df['positive'].values
        df_with_sentiment['sentiment_label'] = sentiment_df.idxmax(axis=1).values
        df_with_sentiment['sentiment_score'] = (
            sentiment_df['positive'] - sentiment_df['negative']
        ).values
        print("Analysis complete!")
        return df_with_sentiment

In [ ]:
# Run sentiment analysis on all documents
analyzer = RedditSentimentAnalyzer()
df_analyzed = analyzer.analyze_dataframe(df, text_column='Document', batch_size=32)

print(f"\nSentiment columns added: {[c for c in df_analyzed.columns if 'sentiment' in c]}")
print(f"\nSample results:")
df_analyzed[['Document', 'subreddit', 'cluster_name', 'sentiment_label', 'sentiment_score']].head(10)

In [ ]:
# Save detailed results
output_file = 'sentiment_by_clusters_detailed.csv'
df_analyzed.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Detailed results saved to: {output_file}")
print(f"Total records: {len(df_analyzed)}")

## 3. Summary Statistics by Cluster

In [ ]:
# Filter to only documents with cluster assignments
df_clustered = df_analyzed[df_analyzed['cluster_name'].notna()].copy()
print(f"Documents with cluster assignments: {len(df_clustered)} / {len(df_analyzed)}")
print()

# Overall summary
print("=" * 60)
print("Overall Sentiment Summary (clustered documents)")
print("=" * 60)
print(f"Total documents:          {len(df_clustered)}")
print(f"Average sentiment score:  {df_clustered['sentiment_score'].mean():.4f}")
print(f"Std sentiment score:      {df_clustered['sentiment_score'].std():.4f}")
print(f"Positive documents:       {(df_clustered['sentiment_label'] == 'positive').sum()} ({(df_clustered['sentiment_label'] == 'positive').mean()*100:.1f}%)")
print(f"Neutral documents:        {(df_clustered['sentiment_label'] == 'neutral').sum()} ({(df_clustered['sentiment_label'] == 'neutral').mean()*100:.1f}%)")
print(f"Negative documents:       {(df_clustered['sentiment_label'] == 'negative').sum()} ({(df_clustered['sentiment_label'] == 'negative').mean()*100:.1f}%)")

In [ ]:
# Per-cluster sentiment summary
print("=" * 80)
print("Sentiment by Cluster")
print("=" * 80)

cluster_summary = df_clustered.groupby('cluster_name').agg(
    n_docs=('Document', 'count'),
    n_subreddits=('subreddit', 'nunique'),
    avg_score=('sentiment_score', 'mean'),
    std_score=('sentiment_score', 'std'),
    avg_positive=('sentiment_positive', 'mean'),
    avg_neutral=('sentiment_neutral', 'mean'),
    avg_negative=('sentiment_negative', 'mean'),
).reset_index()

# Add sentiment label distribution per cluster
label_dist = df_clustered.groupby('cluster_name')['sentiment_label'].value_counts(normalize=True).unstack(fill_value=0)
label_dist.columns = [f'pct_{c}' for c in label_dist.columns]
cluster_summary = cluster_summary.merge(label_dist, left_on='cluster_name', right_index=True, how='left')

cluster_summary = cluster_summary.sort_values('n_docs', ascending=False)

# Save
cluster_summary.to_csv('cluster_sentiment_summary.csv', index=False, encoding='utf-8-sig')
print("Saved to: cluster_sentiment_summary.csv\n")

cluster_summary.round(4)

## 4. Visualizations

### 4.1 Sentiment Distribution Across Clusters (Bar Chart)

In [ ]:
# Stacked bar chart: sentiment composition per cluster
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Sort clusters by avg sentiment score
cs = cluster_summary.sort_values('avg_score', ascending=True).copy()

# --- Plot 1: Stacked bar (negative/neutral/positive probabilities) ---
ax1 = axes[0]
y_pos = range(len(cs))
ax1.barh(y_pos, cs['avg_negative'], color='#e74c3c', label='Negative', alpha=0.85)
ax1.barh(y_pos, cs['avg_neutral'], left=cs['avg_negative'], color='#95a5a6', label='Neutral', alpha=0.85)
ax1.barh(y_pos, cs['avg_positive'], left=cs['avg_negative'] + cs['avg_neutral'],
         color='#2ecc71', label='Positive', alpha=0.85)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(cs['cluster_name'], fontsize=9)
ax1.set_xlabel('Average Probability', fontsize=12)
ax1.set_title('Sentiment Composition by Cluster (Stacked Probabilities)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3, axis='x')

# --- Plot 2: Net sentiment score ---
ax2 = axes[1]
colors = ['#2ecc71' if score >= 0 else '#e74c3c' for score in cs['avg_score']]
bars = ax2.barh(y_pos, cs['avg_score'], color=colors, alpha=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(cs['cluster_name'], fontsize=9)
ax2.set_xlabel('Average Sentiment Score (Positive - Negative)', fontsize=12)
ax2.set_title('Net Sentiment Score by Cluster', fontsize=14, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, score in zip(bars, cs['avg_score']):
    width = bar.get_width()
    label_x = width + 0.01 if width >= 0 else width - 0.01
    ax2.text(label_x, bar.get_y() + bar.get_height()/2, f'{score:.3f}',
             va='center', ha='left' if width >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('sentiment_by_cluster_bars.png', dpi=300, bbox_inches='tight')
print("Saved: sentiment_by_cluster_bars.png")
plt.show()

### 4.2 Sentiment Over Time by Cluster

In [ ]:
# Sentiment over time - aggregated across all clusters
time_col = 'quarter'
time_values = sorted(df_clustered[time_col].dropna().unique())

if len(time_values) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    time_sentiment = df_clustered.groupby(time_col).agg(
        avg_negative=('sentiment_negative', 'mean'),
        avg_neutral=('sentiment_neutral', 'mean'),
        avg_positive=('sentiment_positive', 'mean'),
        avg_score=('sentiment_score', 'mean'),
        n_docs=('Document', 'count')
    ).reindex(time_values).reset_index()

    # Plot 1: Three sentiment probabilities over time
    ax1 = axes[0]
    ax1.plot(time_sentiment[time_col], time_sentiment['avg_negative'],
             marker='o', label='Negative', linewidth=2, color='#e74c3c')
    ax1.plot(time_sentiment[time_col], time_sentiment['avg_neutral'],
             marker='s', label='Neutral', linewidth=2, color='#95a5a6')
    ax1.plot(time_sentiment[time_col], time_sentiment['avg_positive'],
             marker='^', label='Positive', linewidth=2, color='#2ecc71')
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('Average Probability', fontsize=12)
    ax1.set_title('Sentiment Distribution Over Time (All Clusters)', fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Plot 2: Net sentiment score over time
    ax2 = axes[1]
    ax2.plot(time_sentiment[time_col], time_sentiment['avg_score'],
             marker='o', color='purple', linewidth=2)
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax2.fill_between(range(len(time_sentiment)), time_sentiment['avg_score'], 0,
                     where=(time_sentiment['avg_score'] >= 0),
                     alpha=0.3, color='green', label='Positive')
    ax2.fill_between(range(len(time_sentiment)), time_sentiment['avg_score'], 0,
                     where=(time_sentiment['avg_score'] < 0),
                     alpha=0.3, color='red', label='Negative')
    ax2.set_xticks(range(len(time_sentiment)))
    ax2.set_xticklabels(time_sentiment[time_col], rotation=45, ha='right')
    ax2.set_xlabel('Time', fontsize=12)
    ax2.set_ylabel('Sentiment Score (Positive - Negative)', fontsize=12)
    ax2.set_title('Net Sentiment Score Over Time (All Clusters)', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('sentiment_over_time_all_clusters.png', dpi=300, bbox_inches='tight')
    print("Saved: sentiment_over_time_all_clusters.png")
    plt.show()
else:
    print(f"Only {len(time_values)} time period(s) found: {time_values}")
    print("Skipping time-series plot (need at least 2 time periods).")

In [ ]:
# Sentiment over time - per cluster (top clusters by document count)
time_values = sorted(df_clustered[time_col].dropna().unique())

if len(time_values) > 1:
    top_clusters = cluster_summary.nlargest(6, 'n_docs')['cluster_name'].tolist()

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes_flat = axes.flatten()

    for i, cname in enumerate(top_clusters):
        ax = axes_flat[i]
        cdata = df_clustered[df_clustered['cluster_name'] == cname]
        ts = cdata.groupby(time_col)['sentiment_score'].mean().reindex(time_values)

        ax.plot(range(len(ts)), ts.values, marker='o', linewidth=2, color='purple')
        ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        ax.fill_between(range(len(ts)), ts.values, 0,
                        where=(ts.values >= 0), alpha=0.3, color='green')
        ax.fill_between(range(len(ts)), ts.values, 0,
                        where=(ts.values < 0), alpha=0.3, color='red')
        ax.set_xticks(range(len(ts)))
        ax.set_xticklabels(time_values, rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{cname}\n(n={len(cdata)})', fontsize=10, fontweight='bold')
        ax.set_ylabel('Sentiment Score', fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Sentiment Score Over Time - Top 6 Clusters', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('sentiment_over_time_per_cluster.png', dpi=300, bbox_inches='tight')
    print("Saved: sentiment_over_time_per_cluster.png")
    plt.show()
else:
    print("Skipping per-cluster time-series plot (need at least 2 time periods).")

### 4.3 Sentiment Heatmap (Cluster x Time)

In [ ]:
time_values = sorted(df_clustered[time_col].dropna().unique())

if len(time_values) > 1:
    pivot_data = df_clustered.pivot_table(
        values='sentiment_score',
        index='cluster_name',
        columns=time_col,
        aggfunc='mean'
    ).reindex(columns=time_values)

    fig, ax = plt.subplots(figsize=(max(14, len(time_values)*2), max(8, len(pivot_data)*0.6)))
    sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0, ax=ax, cbar_kws={'label': 'Sentiment Score'},
                linewidths=0.5)
    ax.set_title('Sentiment Heatmap: Cluster x Time Period', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time Period', fontsize=12)
    ax.set_ylabel('Cluster', fontsize=12)
    plt.setp(ax.yaxis.get_majorticklabels(), fontsize=9)

    plt.tight_layout()
    plt.savefig('sentiment_heatmap_cluster_time.png', dpi=300, bbox_inches='tight')
    print("Saved: sentiment_heatmap_cluster_time.png")
    plt.show()
else:
    print("Skipping heatmap (need at least 2 time periods).")
    
    # Show a simple heatmap of cluster x sentiment metrics instead
    metrics = cluster_summary[['cluster_name', 'avg_positive', 'avg_neutral', 'avg_negative', 'avg_score']].set_index('cluster_name')
    metrics.columns = ['Positive', 'Neutral', 'Negative', 'Net Score']
    
    fig, ax = plt.subplots(figsize=(10, max(8, len(metrics)*0.5)))
    sns.heatmap(metrics, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0, ax=ax, cbar_kws={'label': 'Value'},
                linewidths=0.5)
    ax.set_title('Sentiment Metrics by Cluster', fontsize=14, fontweight='bold')
    plt.setp(ax.yaxis.get_majorticklabels(), fontsize=9)
    plt.tight_layout()
    plt.savefig('sentiment_heatmap_cluster_metrics.png', dpi=300, bbox_inches='tight')
    print("Saved: sentiment_heatmap_cluster_metrics.png")
    plt.show()

### 4.4 Sentiment Label Distribution by Cluster

In [ ]:
# Sentiment label proportions per cluster
label_by_cluster = df_clustered.groupby('cluster_name')['sentiment_label'].value_counts(normalize=True).unstack(fill_value=0)

# Reorder columns
for col in ['negative', 'neutral', 'positive']:
    if col not in label_by_cluster.columns:
        label_by_cluster[col] = 0
label_by_cluster = label_by_cluster[['negative', 'neutral', 'positive']]

# Sort by positive ratio
label_by_cluster = label_by_cluster.sort_values('positive', ascending=True)

fig, ax = plt.subplots(figsize=(14, max(8, len(label_by_cluster)*0.5)))
label_by_cluster.plot(kind='barh', stacked=True, ax=ax,
                      color=['#e74c3c', '#95a5a6', '#2ecc71'], alpha=0.85)
ax.set_xlabel('Proportion', fontsize=12)
ax.set_title('Sentiment Label Distribution by Cluster', fontsize=14, fontweight='bold')
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, alpha=0.3, axis='x')
plt.setp(ax.yaxis.get_majorticklabels(), fontsize=9)

plt.tight_layout()
plt.savefig('sentiment_labels_by_cluster.png', dpi=300, bbox_inches='tight')
print("Saved: sentiment_labels_by_cluster.png")
plt.show()

### 4.5 Sentiment Score Distribution (Box Plot)

In [ ]:
# Box plot showing distribution of sentiment scores within each cluster
order = cluster_summary.sort_values('avg_score', ascending=False)['cluster_name'].tolist()

fig, ax = plt.subplots(figsize=(14, max(8, len(order)*0.5)))
sns.boxplot(data=df_clustered, y='cluster_name', x='sentiment_score',
            order=order, ax=ax, palette='RdYlGn', orient='h')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Sentiment Score (Positive - Negative)', fontsize=12)
ax.set_ylabel('')
ax.set_title('Sentiment Score Distribution by Cluster', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.setp(ax.yaxis.get_majorticklabels(), fontsize=9)

plt.tight_layout()
plt.savefig('sentiment_boxplot_by_cluster.png', dpi=300, bbox_inches='tight')
print("Saved: sentiment_boxplot_by_cluster.png")
plt.show()

## 5. Quarterly Sentiment Summary by Cluster

In [ ]:
# Quarterly summary per cluster
quarterly_cluster = df_clustered.groupby(['cluster_name', time_col]).agg(
    n_docs=('Document', 'count'),
    avg_score=('sentiment_score', 'mean'),
    std_score=('sentiment_score', 'std'),
    avg_positive=('sentiment_positive', 'mean'),
    avg_neutral=('sentiment_neutral', 'mean'),
    avg_negative=('sentiment_negative', 'mean'),
).reset_index()

quarterly_cluster.to_csv('cluster_quarterly_sentiment.csv', index=False, encoding='utf-8-sig')
print("Saved: cluster_quarterly_sentiment.csv")
print(f"\nShape: {quarterly_cluster.shape}")
quarterly_cluster.head(20)

## 6. Final Summary Report

In [ ]:
print("=" * 60)
print("Analysis Complete - Summary Report")
print("=" * 60)

print(f"\nTotal documents analyzed:       {len(df_analyzed)}")
print(f"Documents with cluster mapping:  {len(df_clustered)}")
print(f"Number of clusters:              {df_clustered['cluster_name'].nunique()}")
print(f"Number of subreddits (matched):  {df_clustered['subreddit'].nunique()}")

print("\n--- Top 5 Most Positive Clusters ---")
top_pos = cluster_summary.nlargest(5, 'avg_score')[['cluster_name', 'avg_score', 'n_docs']]
for _, row in top_pos.iterrows():
    print(f"  {row['cluster_name']:<50s}  score={row['avg_score']:+.4f}  (n={row['n_docs']})")

print("\n--- Top 5 Most Negative Clusters ---")
top_neg = cluster_summary.nsmallest(5, 'avg_score')[['cluster_name', 'avg_score', 'n_docs']]
for _, row in top_neg.iterrows():
    print(f"  {row['cluster_name']:<50s}  score={row['avg_score']:+.4f}  (n={row['n_docs']})")

print("\n--- Output Files ---")
print("  Data:")
print("    - sentiment_by_clusters_detailed.csv      (per-document sentiment with cluster info)")
print("    - cluster_sentiment_summary.csv            (per-cluster aggregated stats)")
print("    - cluster_quarterly_sentiment.csv          (per-cluster per-quarter stats)")
print("  Charts:")
print("    - sentiment_by_cluster_bars.png            (stacked bars + net score)")
print("    - sentiment_over_time_all_clusters.png     (time trend overall)")
print("    - sentiment_over_time_per_cluster.png      (time trend per cluster)")
print("    - sentiment_heatmap_cluster_time.png       (heatmap cluster x time)")
print("    - sentiment_labels_by_cluster.png          (label distribution)")
print("    - sentiment_boxplot_by_cluster.png         (score distribution box plot)")
print("\n" + "=" * 60)
print("Done!")
print("=" * 60)